In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SIMULACIÓN ESTÁNDAR DE LA GEOMETRÍA RELACIONAL (RG)
===================================================
Arquitecto: Edward P. López
Versión: 4.1 – Simulación estándar

Esta es una ejecución única de la simulación Monte Carlo
con los parámetros estándar de la RG.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
from scipy.fftpack import fft2, fftshift
import csv
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONSTANTES DE LA RG
# ============================================================================

D_DEUDA = 1.5
K_RED = 57 / 24.125
N19_TARGET = 19
N57_TARGET = 57
LAMBDA_F = 1 / 18
ETA = np.pi / 57
PI = np.pi
ALPHA_INV_RG = 4*PI**3 + PI**2 + PI
ME_RG = 3 * D_DEUDA / PI
MP_ME_RG = 1836.15

print("="*80)
print("SIMULACIÓN ESTÁNDAR RG")
print(f"Deuda D = {D_DEUDA:.4f} | K_red = {K_RED:.6f} | η = {ETA:.6f}")
print(f"λ = {LAMBDA_F:.6f} | α⁻¹ (RG) = {ALPHA_INV_RG:.6f}")
print("="*80)

# ============================================================================
# CLASE DE SIMULACIÓN
# ============================================================================

class SimulacionRG:
    def __init__(self, size=60, J=1.2, T=0.08, pasos=800, semilla=192657):
        self.size = size
        self.N = size * size
        self.J = J
        self.T = T
        self.pasos = pasos
        self.semilla = semilla
        self.delta_max = 0.15

        np.random.seed(semilla)
        self.phase = np.zeros((size, size))
        self.history_phase = []
        self.history_energy = []
        self.history_tension = []
        self.history_max_cluster = []
        self.history_has_N19 = []
        self.history_has_N57 = []
        self.history_acceptance = []

        self.D = D_DEUDA
        self.K_red = K_RED
        self.N19 = N19_TARGET
        self.N57 = N57_TARGET

        print(f"[RG-MC] Red {size}x{size} = {self.N} nodos")
        print(f"[RG-MC] J={J}, T={T}, pasos={pasos}, semilla={semilla}")

    def potencial(self, theta):
        x = theta / PI
        return (self.D / self.N) * (x**4 - 2*x**2 + 1)

    def obtener_vecinos(self, i, j):
        vecinos = []
        desplazamientos = [(-1,0), (1,0), (0,-1), (0,1), (-1,1), (1,-1)]
        for di, dj in desplazamientos:
            ni, nj = i + di, j + dj
            if 0 <= ni < self.size and 0 <= nj < self.size:
                vecinos.append((ni, nj))
        return vecinos

    def energia_local(self, i, j):
        theta = self.phase[i, j]
        E = self.potencial(theta)
        for ni, nj in self.obtener_vecinos(i, j):
            dtheta = theta - self.phase[ni, nj]
            E += self.J * (1 - np.cos(dtheta))
        return E

    def energia_total(self):
        E = np.sum(self.potencial(self.phase))
        for i in range(self.size):
            for j in range(self.size):
                for ni, nj in self.obtener_vecinos(i, j):
                    if (i, j) < (ni, nj):
                        dtheta = self.phase[i,j] - self.phase[ni,nj]
                        E += self.J * (1 - np.cos(dtheta))
        return E

    def tension_media(self):
        grad = np.gradient(self.phase)
        return np.mean(np.sqrt(grad[0]**2 + grad[1]**2))

    def silencio(self):
        self.phase = np.zeros((self.size, self.size))
        self.history_phase = [self.phase.copy()]
        self.history_energy = [self.energia_total()]
        self.history_tension = [self.tension_media()]
        self.history_max_cluster = [0]
        self.history_has_N19 = [False]
        self.history_has_N57 = [False]
        self.history_acceptance = []
        print("[RG-MC] Silencio Armónico establecido (θ=0)")

    def autocomparacion(self, amplitud=1.5):
        centro = self.size // 2
        self.phase[centro, centro] = amplitud
        self.phase[centro, centro+1] = -amplitud * 0.5
        self.phase[centro+1, centro] = -amplitud * 0.5
        self.phase[centro-1, centro] = amplitud * 0.25
        self.phase[centro, centro-1] = amplitud * 0.25
        print(f"[RG-MC] Auto‑comparación dipolo: A={amplitud}, B={-amplitud*0.5}")

        self.history_phase.append(self.phase.copy())
        self.history_energy.append(self.energia_total())
        self.history_tension.append(self.tension_media())
        self.history_max_cluster.append(0)
        self.history_has_N19.append(False)
        self.history_has_N57.append(False)

    def detectar_clusters(self, umbral=0.3):
        phase_abs = np.abs(self.phase)
        binario = (phase_abs > umbral).astype(int)
        labeled, num_clusters = label(binario, structure=np.ones((3,3)))
        sizes = [np.sum(labeled == i) for i in range(1, num_clusters + 1)]
        return {
            'num': num_clusters,
            'sizes': sizes,
            'max': max(sizes) if sizes else 0,
            'has_N19': N19_TARGET in sizes,
            'has_N57': N57_TARGET in sizes
        }

    def paso_mc(self):
        aceptados = 0
        for _ in range(self.N):
            i = np.random.randint(0, self.size)
            j = np.random.randint(0, self.size)
            old_phase = self.phase[i, j]
            new_phase = old_phase + np.random.uniform(-self.delta_max, self.delta_max)
            old_E = self.energia_local(i, j)
            self.phase[i, j] = new_phase
            new_E = self.energia_local(i, j)
            dE = new_E - old_E
            if dE > 0 and np.random.rand() > np.exp(-dE / self.T):
                self.phase[i, j] = old_phase
            else:
                aceptados += 1

        self.history_phase.append(self.phase.copy())
        self.history_energy.append(self.energia_total())
        self.history_tension.append(self.tension_media())
        clusters = self.detectar_clusters()
        self.history_max_cluster.append(clusters['max'])
        self.history_has_N19.append(clusters['has_N19'])
        self.history_has_N57.append(clusters['has_N57'])
        return aceptados / self.N

    def ejecutar(self, verbose=True):
        if verbose:
            print(f"[RG-MC] Ejecutando {self.pasos} pasos...")
        for step in range(self.pasos):
            tasa = self.paso_mc()
            self.history_acceptance.append(tasa)
            if verbose and (step+1) % 100 == 0:
                E = self.history_energy[-1]
                T = self.history_tension[-1]
                maxc = self.history_max_cluster[-1]
                print(f"  Paso {step+1:4d}: E={E:.4f}, Ω={T:.4f}, max={maxc}, acep={tasa:.2%}")
        if verbose:
            print("[RG-MC] Simulación completada")
        self.analizar_final()
        return self

    def analizar_ABC(self):
        f = fftshift(fft2(self.phase))
        mag = np.abs(f)
        centro = self.size // 2
        A = mag[centro, centro+1] if centro+1 < self.size else 0
        B = mag[centro+1, centro] if centro+1 < self.size else 0
        C = mag[centro+1, centro+1] if centro+1 < self.size else 0

        c_region = self.phase[centro-2:centro+3, centro-2:centro+3]
        c_val = np.mean(c_region) if c_region.size > 0 else 0
        grad = np.gradient(self.phase)
        a_val = np.mean(np.abs(grad[0][centro-2:centro+3, centro-2:centro+3]))
        b_val = np.mean(np.abs(grad[1][centro-2:centro+3, centro-2:centro+3]))

        left = A * B * C
        right = 2 * a_val * b_val * c_val
        error = abs(left - right) / (abs(left) + abs(right) + 1e-10)
        deuda = left - right

        return {'A': A, 'B': B, 'C': C, 'a': a_val, 'b': b_val, 'c': c_val,
                'ABC': left, '2abc': right, 'error': error, 'deuda': deuda}

    def verificar_limite(self):
        omega = self.tension_media()
        return {'omega': omega, 'K_red': self.K_red, 'supera': omega > self.K_red,
                'ratio': omega / self.K_red}

    def analizar_final(self):
        print("\n" + "="*70)
        print("ANÁLISIS FINAL DE LA SIMULACIÓN")
        print("="*70)

        print(f"\n1. ESTADO FINAL")
        print(f"   Fase media: {np.mean(self.phase):.6f}")
        print(f"   Fase RMS: {np.std(self.phase):.6f}")
        print(f"   Energía final: {self.history_energy[-1]:.4f}")
        print(f"   Tensión final: {self.history_tension[-1]:.4f}")

        abc = self.analizar_ABC()
        print(f"\n2. ECUACIÓN MAESTRA (ABC = 2abc)")
        print(f"   A={abc['A']:.4f}, B={abc['B']:.4f}, C={abc['C']:.4f}")
        print(f"   a={abc['a']:.4f}, b={abc['b']:.4f}, c={abc['c']:.4f}")
        print(f"   ABC={abc['ABC']:.4f}, 2abc={abc['2abc']:.4f}")
        print(f"   Deuda = {abc['deuda']:.6f} (esperado D = {self.D:.4f})")
        print(f"   Error relativo = {abc['error']:.6e}")

        clusters = self.detectar_clusters()
        print(f"\n3. CLÚSTERES TOPOLÓGICOS")
        print(f"   Número de clústeres: {clusters['num']}")
        print(f"   Clúster máximo: {clusters['max']}")
        print(f"   N19 presente: {'✓' if clusters['has_N19'] else '✗'}")
        print(f"   N57 presente: {'✓' if clusters['has_N57'] else '✗'}")

        limite = self.verificar_limite()
        print(f"\n4. LÍMITE DE PERMISIBILIDAD")
        print(f"   Ω = {limite['omega']:.6f}")
        print(f"   K_red = {limite['K_red']:.6f}")
        print(f"   Ratio = {limite['ratio']:.6f}")
        print(f"   Supera: {'⚠ SÍ (¡RUPTURA!)' if limite['supera'] else '✓ No (estable)'}")

        print(f"\n5. COMPARACIÓN CON PREDICCIONES RG")
        print(f"   α⁻¹ (RG) = {ALPHA_INV_RG:.6f} (esperado 137.035999084)")
        print(f"   Error α⁻¹ = {abs(ALPHA_INV_RG - 137.035999084):.6f}")
        print(f"   m_e (RG)  = {ME_RG:.4f} u.r. (0.511 MeV)")
        print(f"   m_p/m_e  = {MP_ME_RG:.2f} (esperado 1836.15)")

        print("\n" + "="*70)
        print("FIN DEL ANÁLISIS")
        print("="*70)

    def graficar_estado(self, save=True):
        phase = self.history_phase[-1]
        E = self.history_energy[-1]
        T = self.history_tension[-1]
        abc = self.analizar_ABC()
        clusters = self.detectar_clusters()
        limite = self.verificar_limite()

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Simulación Estándar RG — Estado Final', fontsize=16)

        im1 = axes[0, 0].imshow(phase, cmap='RdBu', vmin=-1.5, vmax=1.5)
        axes[0, 0].set_title('Fase de la Red (θ)')
        axes[0, 0].axis('off')
        plt.colorbar(im1, ax=axes[0, 0])

        axes[0, 1].plot(self.history_energy, 'b-', alpha=0.7, linewidth=0.8)
        axes[0, 1].axhline(y=self.history_energy[-1], color='r', linestyle='--',
                          label=f'E_final = {E:.2f}')
        axes[0, 1].set_xlabel('Pasos de MC')
        axes[0, 1].set_ylabel('Energía Total')
        axes[0, 1].set_title('Evolución de la Energía')
        axes[0, 1].legend()
        axes[0, 1].grid(alpha=0.3)

        axes[0, 2].plot(self.history_tension, 'r-', alpha=0.7, linewidth=0.8)
        axes[0, 2].axhline(y=self.K_red, color='k', linestyle='--',
                          label=f'K_red = {self.K_red:.4f}')
        axes[0, 2].set_xlabel('Pasos de MC')
        axes[0, 2].set_ylabel('Tensión Ω')
        axes[0, 2].set_title('Evolución de la Tensión')
        axes[0, 2].legend()
        axes[0, 2].grid(alpha=0.3)

        if clusters['sizes']:
            axes[1, 0].hist(clusters['sizes'], bins=30, color='skyblue', edgecolor='black')
            axes[1, 0].axvline(x=N19_TARGET, color='r', linestyle='--', label='N19')
            axes[1, 0].axvline(x=N57_TARGET, color='g', linestyle='--', label='N57')
            axes[1, 0].set_xlabel('Tamaño del Clúster')
            axes[1, 0].set_ylabel('Frecuencia')
            axes[1, 0].set_title('Distribución de Clústeres')
            axes[1, 0].legend()
            axes[1, 0].set_xlim(0, max(200, max(clusters['sizes'])+50))

        info_text = (
            f"=== ECUACIÓN MAESTRA ===\n"
            f"A={abc['A']:.4f}, B={abc['B']:.4f}, C={abc['C']:.4f}\n"
            f"a={abc['a']:.4f}, b={abc['b']:.4f}, c={abc['c']:.4f}\n"
            f"ABC={abc['ABC']:.4f}\n"
            f"2abc={abc['2abc']:.4f}\n"
            f"Deuda = {abc['deuda']:.6f}\n"
            f"Error = {abc['error']:.6e}\n\n"
            f"=== CLÚSTERES ===\n"
            f"Clústeres: {clusters['num']}\n"
            f"Máximo: {clusters['max']}\n"
            f"N19: {'✓' if clusters['has_N19'] else '✗'}\n"
            f"N57: {'✓' if clusters['has_N57'] else '✗'}\n\n"
            f"=== LÍMITE ===\n"
            f"Ω={limite['omega']:.6f}\n"
            f"K_red={limite['K_red']:.6f}\n"
            f"Ratio={limite['ratio']:.6f}\n"
            f"Supera: {'⚠' if limite['supera'] else '✓'}"
        )

        axes[1, 1].text(0.1, 0.95, info_text, transform=axes[1, 1].transAxes,
                       fontsize=10, verticalalignment='top', fontfamily='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        axes[1, 1].axis('off')
        axes[1, 1].set_title('Diagnóstico RG')

        axes[1, 2].plot(self.history_max_cluster, 'g-', alpha=0.7, linewidth=0.8)
        axes[1, 2].axhline(y=N19_TARGET, color='r', linestyle='--', label='N19')
        axes[1, 2].axhline(y=N57_TARGET, color='g', linestyle='--', label='N57')
        axes[1, 2].set_xlabel('Pasos de MC')
        axes[1, 2].set_ylabel('Clúster Máximo')
        axes[1, 2].set_title('Evolución del Clúster Máximo')
        axes[1, 2].legend()
        axes[1, 2].grid(alpha=0.3)

        plt.tight_layout()
        if save:
            plt.savefig(f'rg_estandar_{self.semilla}.png', dpi=150, bbox_inches='tight')
            print(f"[RG-MC] Figura guardada como 'rg_estandar_{self.semilla}.png'")
        plt.show()
        return fig

    def guardar_resultados(self, filename=None):
        if filename is None:
            filename = f'rg_estandar_{self.semilla}.csv'
        min_len = min(len(self.history_energy), len(self.history_tension),
                      len(self.history_max_cluster), len(self.history_has_N19),
                      len(self.history_has_N57))
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Paso', 'Energia', 'Tension', 'MaxCluster', 'N19', 'N57'])
            for i in range(min_len):
                writer.writerow([i, self.history_energy[i], self.history_tension[i],
                                 self.history_max_cluster[i], self.history_has_N19[i],
                                 self.history_has_N57[i]])
        print(f"[RG-MC] Resultados guardados en '{filename}'")


# ============================================================================
# EJECUCIÓN DE LA SIMULACIÓN ESTÁNDAR
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("EJECUTANDO SIMULACIÓN ESTÁNDAR RG")
    print("="*80)

    sim = SimulacionRG(size=60, J=1.2, T=0.08, pasos=800, semilla=192657)
    sim.silencio()
    sim.autocomparacion(amplitud=1.5)
    sim.ejecutar(verbose=True)
    sim.graficar_estado(save=True)
    sim.guardar_resultados()

    print("\n" + "="*80)
    print("SIMULACIÓN ESTÁNDAR COMPLETADA")
    print("="*80)